In [ ]:
import os
import csv
import requests
import time
import random


GITHUB_TOKEN = ""

TOTAL_REPOS_TO_FIND = 2000
CSV_FILE = 'individual_repositories.csv'
API_URL = 'https://api.github.com/search/repositories'

# Skipping for known organization/foundation account names
EXCLUDED_OWNERS = {
    'google', 'microsoft', 'facebook', 'apple', 'amazon', 'netflix', 'ibm',
    'oracle', 'intel', 'adobe', 'airbnb', 'uber', 'linkedin', 'twitter',
    'mozilla', 'apache', 'torvalds', 'docker', 'kubernetes', 'tensorflow',
    'pytorch', 'angular', 'vuejs', 'reactjs', 'nodejs', 'golang', 'rust-lang',
    'jetbrains', 'elastic', 'mongodb', 'automattic', 'square', 'shopify',
    'stripe', 'spotify', 'dropbox', 'github', 'gitlab', 'atlassian', 'slack',
    'zoom', 'salesforce', 'unity', 'unreal', 'epic', 'valve', 'steam'
}

def check_rate_limit():
    """Check remaining rate limit."""
    headers = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}
    try:
        response = requests.get('https://api.github.com/rate_limit', headers=headers)
        if response.status_code == 200:
            data = response.json()
            search_remaining = data['resources']['search']['remaining']
            search_reset = data['resources']['search']['reset']
            print(f"Search API remaining: {search_remaining}")
            return search_remaining, search_reset
        return 0, 0
    except:
        return 0, 0

def wait_for_rate_limit_reset(reset_time):
    """Wait until rate limit resets."""
    current_time = time.time()
    wait_time = max(0, reset_time - current_time + 10)  # Add 10 seconds buffer
    if wait_time > 0:
        print(f"Rate limit exceeded. Waiting {wait_time:.0f} seconds...")
        time.sleep(wait_time)

def search_github_repos(query, page):
    """Sends a search request to the GitHub API with better error handling."""
    headers = {
        'Accept': 'application/vnd.github.v3+json',
        'User-Agent': 'GitHub-Repo-Scraper'
    }
    if GITHUB_TOKEN:
        headers['Authorization'] = f'token {GITHUB_TOKEN}'
    
    params = {
        'q': query,
        'sort': 'stars',
        'order': 'desc',
        'per_page': 100,
        'page': page
    }
    
    try:
        response = requests.get(API_URL, headers=headers, params=params)
        
        if response.status_code == 403:
            remaining, reset_time = check_rate_limit()
            if remaining == 0:
                wait_for_rate_limit_reset(reset_time)
                # Retry the request
                response = requests.get(API_URL, headers=headers, params=params)
            else:
                print(f"403 error but rate limit shows {remaining} remaining. Waiting 60 seconds...")
                time.sleep(60)
                return None
        
        response.raise_for_status()
        return response.json()
        
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None

def generate_diverse_queries():
    """Generate more diverse queries to find different types of repositories."""
    queries = []
    
    # Popular languages
    languages = ['python', 'javascript', 'java', 'cpp', 'csharp']
    
    # Different approaches to find individual repos
    base_queries = [
        # Language-based queries (more likely to find individual developers)
        *[f'language:{lang} type:user stars:10..500' for lang in languages],
        *[f'language:{lang} type:user stars:5..50 forks:1..20' for lang in languages[:5]],
        
        # Topic-based queries
        'topic:personal-project type:user stars:5..100',
        'topic:learning type:user stars:1..50',
        'topic:portfolio type:user stars:1..100',
        'topic:tutorial type:user stars:10..200',
        'topic:practice type:user stars:1..50',
        
        # Time-based but broader
        'type:user created:2023-01-01..2024-12-31 stars:5..100',
        'type:user created:2022-01-01..2023-12-31 stars:10..200',
        'type:user created:2021-01-01..2022-12-31 stars:15..300',
        'type:user created:2020-01-01..2021-12-31 stars:20..400',
        
        # Size-based queries (smaller repos more likely individual)
        'type:user size:1..1000 stars:5..100',
        'type:user size:1000..10000 stars:10..200',
        
        # Fork-based queries
        'type:user forks:1..10 stars:5..100',
        'type:user forks:0..5 stars:10..200',
        
        # Combination queries
        'type:user stars:1..20 language:python',
        'type:user stars:1..20 language:javascript',
        'type:user stars:5..50 language:java',
        'type:user stars:10..100 language:cpp',
        
        # Broader searches
        'type:user stars:1..100',
        'type:user stars:5..200',
        'type:user forks:1..50',
    ]
    
    # Shuffle to add randomness
    random.shuffle(base_queries)
    return base_queries

def main():
    """Main function to find repos and write to CSV."""
    if not GITHUB_TOKEN:
        print("Error: GITHUB_TOKEN environment variable not set.")
        return

    print(f"Starting repository search. Goal: {TOTAL_REPOS_TO_FIND} repositories.")
    
    # Check initial rate limit
    remaining, reset_time = check_rate_limit()
    print(f"Starting with {remaining} API calls remaining")

    repo_urls = set()
    found_repos = []
    file_exists = os.path.isfile(CSV_FILE)

    # Load existing repos
    if file_exists:
        print(f"Reading existing repositories from {CSV_FILE}...")
        with open(CSV_FILE, 'r', newline='', encoding='utf-8') as csvfile:
            try:
                csv_reader = csv.reader(csvfile)
                header = next(csv_reader)
                for row in csv_reader:
                    if len(row) > 2:
                        repo_urls.add(row[2])
                        found_repos.append(row)
            except StopIteration:
                pass
        print(f"Found {len(found_repos)} existing repositories.")

    if len(found_repos) >= TOTAL_REPOS_TO_FIND:
        print(f"Already have {len(found_repos)} repositories. Exiting.")
        return

    queries = generate_diverse_queries()
    print(f"Generated {len(queries)} diverse queries")

    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        
        if not file_exists or os.path.getsize(CSV_FILE) == 0:
            csv_writer.writerow(['Owner account Name', 'Repo Name', 'Link', 'Language of repo'])

        target_reached = False
        consecutive_failures = 0
        
        for query_idx, query in enumerate(queries):
            if target_reached:
                break
            
            print(f"\nQuery {query_idx + 1}/{len(queries)}: '{query}'")
            print(f"Current count: {len(found_repos)}/{TOTAL_REPOS_TO_FIND}")
            
            # Check rate limit before starting new query
            remaining, reset_time = check_rate_limit()
            if remaining < 5:  # Leave some buffer
                print("Low on API calls, waiting for reset...")
                wait_for_rate_limit_reset(reset_time)
            
            query_found_new = False
            
            for page in range(1, 6):  # Reduced to 5 pages per query for diversity
                if target_reached:
                    break

                print(f"  Page {page}...")
                data = search_github_repos(query, page)

                if not data:
                    consecutive_failures += 1
                    print(f"  Failed to get data (consecutive failures: {consecutive_failures})")
                    if consecutive_failures >= 5:
                        print("Too many consecutive failures, taking a longer break...")
                        time.sleep(300)  # 5 minute break
                        consecutive_failures = 0
                    break

                consecutive_failures = 0

                if 'items' not in data or not data['items']:
                    print(f"  No items on page {page}")
                    break

                new_repos_this_page = 0
                for item in data['items']:
                    if target_reached:
                        break
                        
                    owner_name = item['owner']['login']
                    owner_type = item['owner']['type']
                    
                    # More strict filtering
                    if (item['html_url'] not in repo_urls and 
                        owner_name.lower() not in EXCLUDED_OWNERS and
                        owner_type == 'User'):  # Only individual users
                        
                        repo_name = item['name']
                        link = item['html_url']
                        language = item['language'] or 'N/A'
                        
                        repo_data = [owner_name, repo_name, link, language]
                        found_repos.append(repo_data)
                        repo_urls.add(link)
                        csv_writer.writerow(repo_data)
                        csvfile.flush()  # Ensure data is written immediately
                        new_repos_this_page += 1
                        query_found_new = True

                        if len(found_repos) % 25 == 0:
                            print(f"    Progress: {len(found_repos)}/{TOTAL_REPOS_TO_FIND} repositories")
                        
                        if len(found_repos) >= TOTAL_REPOS_TO_FIND:
                            target_reached = True
                            break

                print(f"  Added {new_repos_this_page} new repos from page {page}")
                
                if new_repos_this_page == 0:
                    break  # No point checking more pages if no new repos

                # Respectful delay
                time.sleep(random.uniform(3, 6))

            if not query_found_new:
                print(f"  No new repositories found for this query")
            
            # Longer delay between queries
            time.sleep(random.uniform(5, 10))

    print(f"\nFinished! Total repositories found: {len(found_repos)}")
    print(f"Data saved to {CSV_FILE}")
    
    # Final rate limit check
    remaining, _ = check_rate_limit()
    print(f"Remaining API calls: {remaining}")

if __name__ == '__main__':
    main()

Starting repository search. Goal: 2000 repositories.
Search API remaining: 30
Starting with 30 API calls remaining
Generated 30 diverse queries

Query 1/30: 'topic:practice type:user stars:1..50'
Current count: 0/2000
Search API remaining: 30
  Page 1...
  Added 1 new repos from page 1
  Page 2...
  No items on page 2

Query 2/30: 'type:user created:2021-01-01..2022-12-31 stars:15..300'
Current count: 1/2000
Search API remaining: 28
  Page 1...
    Progress: 25/2000 repositories
    Progress: 50/2000 repositories
  Added 67 new repos from page 1
  Page 2...
  No items on page 2

Query 3/30: 'language:javascript type:user stars:10..500'
Current count: 68/2000
Search API remaining: 26
  Page 1...
    Progress: 75/2000 repositories
    Progress: 100/2000 repositories
    Progress: 125/2000 repositories
  Added 60 new repos from page 1
  Page 2...
  No items on page 2

Query 4/30: 'type:user size:1..1000 stars:5..100'
Current count: 128/2000
Search API remaining: 24
  Page 1...
    Progres

KeyboardInterrupt: 